In [1]:
from dataclasses import asdict, dataclass

from areal.api.cli_args import GRPOConfig, load_expr_config

args = ["--config", "AReaL/examples/lite/configs/clevr_count_70k_grpo_modified.yaml"]
config, _ = load_expr_config(args, GRPOConfig)
config: GRPOConfig

/home/aiscuser/.conda/envs/areal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-08-14 11:41:34,299	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
config

GRPOConfig(experiment_name='clevr_count_70k-grpo', trial_name='trial1', cluster=ClusterSpecConfig(name_resolve=NameResolveConfig(type='nfs', nfs_record_root='/tmp/experiment/name_resolve/clevr_count_70k-grpo', etcd3_addr='etcd-client.openpsi-etcd.svc.sigma-na130-lingbo.na130.wl-robby.local:2379', ray_actor_name='ray_kv_store'), cluster_name='na132', fileroot='/tmp/experiments', n_nodes=1, n_gpus_per_node=8), allocation_mode='sglang.d1p1t1+d7p1t1', seed=1, total_train_epochs=3, total_train_steps=None, total_train_n_seqs=None, tokenizer_path='Qwen/Qwen2.5-VL-3B-Instruct', train_dataset=DatasetConfig(path='/storage/openpsi/data/clevr_count_70k/', type='rl', batch_size=32, shuffle=True, pin_memory=True, num_workers=4, drop_last=True, reward_fn=None), valid_dataset=DatasetConfig(path='/storage/openpsi/data/clevr_count_70k/', type='rl', batch_size=32, shuffle=True, pin_memory=True, num_workers=4, drop_last=True, reward_fn=None), saver=SaverConfig(experiment_name='clevr_count_70k-grpo', trial

In [3]:
from areal.utils.network import find_free_ports

SGLANG_PORT, MASTER_PORT = find_free_ports(2)
SGLANG_HOST = "127.0.0.1"

# Environment variables used by inference/train engines
import os

os.environ["AREAL_LLM_SERVER_ADDRS"] = f"{SGLANG_HOST}:{SGLANG_PORT}"
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = str(MASTER_PORT)
os.environ["RANK"] = str(0)
os.environ["WORLD_SIZE"] = str(1)
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["LOCAL_RANK"] = str(0)

In [4]:
import subprocess
import sys

# 启动sglang server
from areal.api.cli_args import SGLangConfig
from areal.utils.network import find_free_ports

config.sglang.log_level = "info"
config.sglang.decode_log_interval = 10
sglang_cmd = SGLangConfig.build_cmd(
    config.sglang,
    tp_size=1,
    base_gpu_id=1,
    host=SGLANG_HOST,
    port=SGLANG_PORT,
)
sglang_process = subprocess.Popen(
    sglang_cmd,
    shell=True,
    stdout=sys.stdout,
    stderr=sys.stderr,
)

In [5]:
# load gsm8k dataset
from datasets import load_dataset
from areal.dataset.clevr_count_70k import get_clevr_count_70k_rl_dataset

In [6]:
import asyncio
import functools
import os
import time
import uuid

import colorama
import torch
from tensordict import TensorDict
from transformers import AutoTokenizer, PreTrainedTokenizerFast

from areal.api.cli_args import GenerationHyperparameters
from areal.api.engine_api import InferenceEngine
from areal.api.io_struct import (
    AllocationMode,
    FinetuneSpec,
    ModelRequest,
    WeightUpdateMeta,
)
from areal.api.workflow_api import RolloutWorkflow
from areal.engine.ppo.actor import FSDPPPOActor
from areal.engine.sglang_remote import RemoteSGLangEngine
from areal.utils.data import concat_padded_tensors
from areal.utils.device import log_gpu_stats
from realhf.api.core.data_api import load_hf_processor_and_tokenizer

processor, tokenizer = load_hf_processor_and_tokenizer(config.tokenizer_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
dataset = get_clevr_count_70k_rl_dataset(path="BUAADreamer/clevr_count_70k", split="test",processor=processor,rank=0, world_size=1)

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


In [7]:
from torchdata.stateful_dataloader import StatefulDataLoader

dataloader = StatefulDataLoader(
    dataset,
    batch_size=config.train_dataset.batch_size,
    shuffle=True,
    collate_fn=lambda x: x,
    drop_last=True,
)
from itertools import cycle

data_generator = cycle(dataloader)

ft_spec = FinetuneSpec(
    total_train_epochs=config.total_train_epochs,
    dataset_size=len(dataloader) * config.train_dataset.batch_size,
    train_batch_size=config.train_dataset.batch_size,
)

x = next(data_generator)
print(f">>> The type of a batch is: {type(x)}\n")
print(f">>> Each piece of data has keys: {x[0].keys()}\n")
print(f">>> Example rollout input: {x[0]['messages']}\n")

>>> The type of a batch is: <class 'list'>

>>> Each piece of data has keys: dict_keys(['answer', 'images', 'messages'])

>>> Example rollout input: <|im_start|>system
Solve the following question: count the number of items in the image and provide the final answer in [ ] format, ensuring that only the number is inside the brackets without any additional text or explanations. <|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>How many  items are there in the image?<|im_end|>
<|im_start|>assistant




In [8]:
import os
import re
import sys
def extract_answer(pred_str, data_name, use_last_number=True):
    match = re.findall(r"\[([0-9\.]+)\]", pred_str)
    if match:
        return match[-1]

    return ""


def clevr_count_70k_reward_fn(
    prompt, completions, prompt_ids, completion_ids, answer, **kwargs
):
    sol = extract_answer(completions, data_name="")  # str number
    ans = answer

    if sol is None:
        return 0
    if ans is None:
        return 0

    if sol.strip() == ans.strip():
        print(f"completions: {completions}, answer: {answer}")
        return 1

    return 0

In [9]:
import asyncio
import os
import uuid

import colorama
import torch
from tensordict import TensorDict
from transformers import AutoProcessor, PreTrainedTokenizerFast

from areal.api.cli_args import GenerationHyperparameters
from areal.api.io_struct import ModelRequest
from areal.utils.data import concat_padded_tensors
from areal.utils.image import image2base64
from areal.workflow.rlvr import RLVRWorkflow
from realhf.base import logging
class VisionRLVRWorkflow(RLVRWorkflow):
    def __init__(
        self,
        reward_fn,
        gconfig: GenerationHyperparameters,
        tokenizer: PreTrainedTokenizerFast,
        processor: AutoProcessor,
        enable_thinking: bool,
        dump_dir: str | None = None,
    ):
        super().__init__(reward_fn, gconfig, tokenizer, enable_thinking, dump_dir)
        self.processor = processor

    async def arun_episode(self, engine, data):

        processed_input = self.processor(
            images=data["images"],
            text=data["messages"],
            padding=False,
            return_tensors="pt",
        )

        input_ids = processed_input["input_ids"].tolist()[0]

        n_samples = self.gconfig.n_samples

        byte_images = image2base64(data["images"])

        req = ModelRequest(
            rid=uuid.uuid4().hex,
            input_ids=input_ids,
            image_data=byte_images,
            gconfig=self.gconfig.new(n_samples=1),
            tokenizer=self.tokenizer,
            processor=self.processor,
        )
        resps = await asyncio.gather(*[engine.agenerate(req) for _ in range(n_samples)])

        version = engine.get_version()
        prompt_strs = []
        completions_strs = []
        rewards = []
        seqlens = []

        results = []
        asyncio.get_event_loop()
        for resp in resps:
            seq = resp.input_tokens + resp.output_tokens
            logprobs = [0.0] * resp.input_len + resp.output_logprobs
            loss_mask = [0] * resp.input_len + [1] * resp.output_len
            versions = [-1] * resp.input_len + resp.output_versions

            prompt_str = self.tokenizer.decode(input_ids)
            completions_str = self.tokenizer.decode(resp.output_tokens)
            prompt_strs.append(prompt_str)
            completions_strs.append(completions_str)
            seqlens.append(len(seq))
            reward = await self.async_reward_fn(
                prompt=prompt_str,
                completions=completions_str,
                prompt_ids=resp.input_tokens,
                completion_ids=resp.output_tokens,
                **data,
            )
            rewards.append(reward)
            res = dict(
                # unsqueeze to add an additional batch dimension
                input_ids=torch.tensor(seq).unsqueeze(0),
                loss_mask=torch.tensor(loss_mask).unsqueeze(0),
                pixel_values=processed_input["pixel_values"].unsqueeze(0),
                image_grid_thw=processed_input["image_grid_thw"].unsqueeze(0),
                logprobs=torch.tensor(logprobs).unsqueeze(0),
                versions=torch.tensor(versions).unsqueeze(0),
                attention_mask=torch.ones(len(seq), dtype=torch.bool).unsqueeze(0),
                # reward
                rewards=torch.tensor([reward]),
            )
            
            results.append(TensorDict(res, batch_size=[1]))
        if self.dump_dir is not None:
            os.makedirs(os.path.join(self.dump_dir, str(version)), exist_ok=True)
            # Get the unique identifier for this prompt
            qid = None
            for key in ["query_id", "id", "qid"]:
                qid = data.get(key, None)
                if qid is not None:
                    break
            qid = qid or uuid.uuid4().hex

            # Dump rollout to file
            with open(
                os.path.join(self.dump_dir, str(version), f"{qid}.txt"), "a"
            ) as f:
                n_samples = self.gconfig.n_samples
                for i, (p, c, r, sl) in enumerate(
                    zip(prompt_strs, completions_strs, rewards, seqlens)
                ):
                    info = "\n".join(
                        [
                            f"idx: {i + 1} / {n_samples}, seqlen: {sl}, reward is {r}.",
                            f"prompt is \n{colorama.Fore.YELLOW + colorama.Style.DIM}{p}{colorama.Style.RESET_ALL}",
                            f"sequence is: \n{colorama.Fore.YELLOW + colorama.Style.DIM}{c}{colorama.Style.RESET_ALL}",
                        ]
                    )
                    f.write(info + "\n")
        print(results)
        return concat_padded_tensors(results)


In [10]:
# initialize inference engine
from areal.engine.sglang_remote import RemoteSGLangEngine
rollout = RemoteSGLangEngine(config.rollout)
rollout.initialize(None, None)
try:
    # TODO: create workflow
    workflow = VisionRLVRWorkflow(
        reward_fn=clevr_count_70k_reward_fn,
        gconfig=GenerationHyperparameters(n_samples=3,max_new_tokens=512),
        tokenizer=tokenizer,
        processor=processor,
        dump_dir="./test",
        enable_thinking=False,
    )
    sample_data = next(data_generator)[:2]
    res = rollout.rollout_batch(sample_data, workflow=workflow)
    print(res)
finally:
    rollout.destroy()

20250814-11:41:46.048 areal.engine.sglang_remote INFO: Waiting for server ready...
20250814-11:42:04.083 areal.engine.sglang_remote INFO: Servers are all ready!
completions: [5]<|im_end|>, answer: 5
completions: [3]<|im_end|>, answer: 3
completions: [5]<|im_end|>, answer: 5
[TensorDict(
    fields={
        attention_mask: Tensor(shape=torch.Size([1, 208]), device=cpu, dtype=torch.bool, is_shared=False),
        image_grid_thw: Tensor(shape=torch.Size([1, 1, 3]), device=cpu, dtype=torch.int64, is_shared=False),
        input_ids: Tensor(shape=torch.Size([1, 208]), device=cpu, dtype=torch.int64, is_shared=False),
        logprobs: Tensor(shape=torch.Size([1, 208]), device=cpu, dtype=torch.float32, is_shared=False),
        loss_mask: Tensor(shape=torch.Size([1, 208]), device=cpu, dtype=torch.int64, is_shared=False),
        pixel_values: Tensor(shape=torch.Size([1, 560, 1176]), device=cpu, dtype=torch.float32, is_shared=False),
        rewards: Tensor(shape=torch.Size([1]), device=cpu, 